# Indiana 211 Benchmark Data Cleaning

Goal: build a benchmark-ready resource pool for evaluating LLM tool calling over human-service resources.

This notebook intentionally keeps the cleaning logic simple and auditable:

1. Define a curated benchmark category system from Indiana 211 subcategories.
2. Keep only records with complete core fields, a usable address, and at least one contact path.
3. Exclude subcategories that are less suitable for stage-1 user-need matching.
4. Sample as evenly as possible across benchmark categories and original subcategories while limiting repeated service names, sites, and counties.
5. Write a CSV pool and JSONL resource index for benchmark generation and agent retrieval.


## Design Choices

- **Resource unit:** one row from `indiana211_resources_deduped.csv`, identified by `agency_id + site_id + service_name`.
- **Core completeness:** required for reliable GT construction: IDs, names, service labels, taxonomy/subcategory, service area, eligibility/details, and city.
- **Address requirement:** require `address_1`, `city`, `state_province`, and `zipcode`. This favors local, physically actionable referrals for stage 1.
- **Contact requirement:** require at least one of `site_number` or `service_website`, so every GT resource can be recommended with an action path.
- **Curated categories:** original Indiana 211 subcategories are mapped into benchmark-friendly need categories. Directory-like categories are left out for now.
- **Sampling goal:** balance need coverage first, then reduce repetition by capping repeated service names and sites.


In [1]:
from __future__ import annotations

import csv
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

SOURCE_CSV = ROOT / "data/indiana211/indiana211_resources_deduped.csv"
OUT_DIR = ROOT / "data/indiana211/benchmark_curated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PER_SUBCATEGORY = 20
MAX_PER_SERVICE_NAME = 5
MAX_PER_SITE = 2
MAX_PER_COUNTY_PER_BENCHMARK_CATEGORY = 12

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def write_csv(path: Path, rows: list[dict[str, object]]) -> None:
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

def write_jsonl(path: Path, rows: list[dict[str, object]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + "\n")

def split_multi(value: str) -> list[str]:
    return [part.strip() for part in (value or "").split(";") if part.strip()]

def normalize_text(value: object) -> str:
    if value is None:
        return ""
    return " ".join(str(value).split())

def normalize_url(value: str) -> str:
    value = normalize_text(value)
    if not value:
        return ""
    if "@" in value or value.startswith(("http://", "https://")):
        return value
    return "https://" + value.lstrip("/")

def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")

def show_counter(title: str, counter: Counter, n: int = 20) -> None:
    print(f"\n{title}")
    for key, value in counter.most_common(n):
        print(f"{value:>5}  {key}")


## 1. Define Curated Benchmark Categories

The source has 53 subcategories. For stage 1, we keep subcategories that naturally correspond to user help-seeking needs. We intentionally skip directory-like or administrative categories for now, because they tend to produce ambiguous GT pairs.


In [2]:
CURATED_CATEGORY_MAP = {
    "Food": [
        "Food",
    ],
    "Housing & Shelter": [
        "Housing/Shelter",
    ],
    "Utilities & Financial Assistance": [
        "Utilities",
        "Temporary Financial Assistance",
        "Money Management",
    ],
    "Material Goods": [
        "Material Goods",
    ],
    "Transportation": [
        "Transportation",
    ],
    "Benefits & Public Assistance": [
        "Public Assistance Programs",
        "Social Insurance Programs",
        "Tax Organizations and Services",
    ],
    "Health Care": [
        "Emergency Medical Care",
        "Health Screening/Diagnostic Services",
        "Health Supportive Services",
        "Human Reproduction",
        "Outpatient Health Facilities",
        "Rehabilitation/Habilitation Services",
        "Specialized Treatment and Prevention",
        "Specialty Medicine",
    ],
    "Mental Health & Substance Use": [
        "Mental Health Assessment and Treatment",
        "Mental Health Care Facilities",
        "Mental Health Support Services",
        "Mutual Support",
        "Substance Use Disorder Services",
    ],
    "Family & Community Support": [
        "Educational Programs",
        "Educational Support Services",
        "Individual and Family Support Services",
        "Leisure Activities/Recreation",
        "Military Service",
        "Social Development and Enrichment",
    ],
    "Legal, Employment & Consumer Help": [
        "Consumer Assistance and Protection",
        "Employment",
        "Legal Services",
    ],
    "Disaster & Safety Needs": [
        "Disaster Services",
        "Public Health",
    ],
}

SUBCATEGORY_TO_BENCHMARK_CATEGORY = {
    subcategory: benchmark_category
    for benchmark_category, subcategories in CURATED_CATEGORY_MAP.items()
    for subcategory in subcategories
}

CURATED_SUBCATEGORIES = set(SUBCATEGORY_TO_BENCHMARK_CATEGORY)

# Optional benchmark-hygiene exclusion. These rows can be useful in a directory,
# but they tend to create weak stage-1 user-need GT pairs.
EXCLUDE_DIRECTORY_LIKE_SERVICE_NAMES = True
DIRECTORY_LIKE_SERVICE_NAMES = {
    "911 Services",
    "Assessor",
    "Auditor",
    "Commissioner",
    "Commissioners",
    "County Commissioner",
    "Highway Department",
    "Information and Referral",
    "Police Department",
    "Public Library",
    "Recorder",
    "Sandbags",
    "Surveyor",
    "Treasurer",
}

print("benchmark categories:", len(CURATED_CATEGORY_MAP))
print("curated source subcategories:", len(CURATED_SUBCATEGORIES))
for category, subcategories in CURATED_CATEGORY_MAP.items():
    print(f"- {category}: {len(subcategories)} subcategories")


benchmark categories: 11
curated source subcategories: 34
- Food: 1 subcategories
- Housing & Shelter: 1 subcategories
- Utilities & Financial Assistance: 3 subcategories
- Material Goods: 1 subcategories
- Transportation: 1 subcategories
- Benefits & Public Assistance: 3 subcategories
- Health Care: 8 subcategories
- Mental Health & Substance Use: 5 subcategories
- Family & Community Support: 6 subcategories
- Legal, Employment & Consumer Help: 3 subcategories
- Disaster & Safety Needs: 2 subcategories


## 2. Load and Normalize Rows

Normalization here is deliberately conservative: trim whitespace, normalize URLs, split multi-value fields, add a stable `resource_id`, and attach the curated category labels.


In [3]:
raw_rows = read_csv(SOURCE_CSV)

def normalize_row(row: dict[str, str]) -> dict[str, object]:
    normalized = {key: normalize_text(value) for key, value in row.items()}
    normalized["service_website"] = normalize_url(str(normalized.get("service_website", "")))
    normalized["resource_id"] = f"in211-{normalized['agency_id']}-{normalized['site_id']}-{slugify(normalized['service_name'])}"
    source_subcategories = split_multi(str(normalized.get("subcategories", "")))
    benchmark_categories = sorted(
        {
            SUBCATEGORY_TO_BENCHMARK_CATEGORY[sub]
            for sub in source_subcategories
            if sub in SUBCATEGORY_TO_BENCHMARK_CATEGORY
        }
    )
    curated_subcategories = sorted(sub for sub in source_subcategories if sub in CURATED_SUBCATEGORIES)
    normalized["benchmark_categories"] = "; ".join(benchmark_categories)
    normalized["curated_subcategories"] = "; ".join(curated_subcategories)
    normalized["primary_benchmark_category"] = benchmark_categories[0] if benchmark_categories else ""
    normalized["primary_curated_subcategory"] = curated_subcategories[0] if curated_subcategories else ""
    return normalized

rows = [normalize_row(row) for row in raw_rows]
print("loaded rows:", len(rows))
print("example resource_id:", rows[0]["resource_id"])


loaded rows: 9987
example resource_id: in211-11185-75042-harm-reduction-services


## 3. Filter for Benchmark-Ready Records

Hard filters:

1. **Core fields complete:** enough information to define the service, location, category, eligibility, and application process.
2. **Address present:** enough location data for local referrals.
3. **At least one contact path:** phone or website.
4. **In curated category set:** skip less useful stage-1 categories for now.
5. **Optional directory-like service-name exclusion:** remove rows such as Auditor, Treasurer, Public Library, and Police Department when building a user-need benchmark.

This keeps the filtering explainable. We keep rejection reasons for audit rather than silently dropping rows.


In [4]:
CORE_FIELDS = [
    "agency_id",
    "site_id",
    "agency_name",
    "site_name",
    "service_name",
    "taxonomy_categories",
    "subcategories",
    "counties_served",
    "site_eligibility",
    "site_details",
    "city",
]

ADDRESS_FIELDS = ["address_1", "city", "state_province", "zipcode"]

def rejection_reasons(row: dict[str, object]) -> list[str]:
    reasons = []
    missing_core = [field for field in CORE_FIELDS if not row.get(field)]
    if missing_core:
        reasons.append("missing_core:" + ",".join(missing_core))
    missing_address = [field for field in ADDRESS_FIELDS if not row.get(field)]
    if missing_address:
        reasons.append("missing_address:" + ",".join(missing_address))
    if not row.get("site_number") and not row.get("service_website"):
        reasons.append("missing_contact")
    if not row.get("benchmark_categories"):
        reasons.append("not_in_curated_categories")
    if EXCLUDE_DIRECTORY_LIKE_SERVICE_NAMES and row.get("service_name") in DIRECTORY_LIKE_SERVICE_NAMES:
        reasons.append("directory_like_service_name")
    return reasons

annotated = []
for row in rows:
    row = dict(row)
    reasons = rejection_reasons(row)
    row["cleaning_rejection_reasons"] = "; ".join(reasons)
    row["cleaning_pass"] = "yes" if not reasons else "no"
    annotated.append(row)

eligible = [row for row in annotated if row["cleaning_pass"] == "yes"]

print("raw rows:", len(raw_rows))
print("eligible rows:", len(eligible))
reason_counts = Counter(reason for row in annotated for reason in split_multi(row["cleaning_rejection_reasons"]))
show_counter("rejection reasons", reason_counts, 20)
show_counter("eligible by benchmark category", Counter(row["primary_benchmark_category"] for row in eligible), 20)
show_counter("eligible by curated subcategory", Counter(row["primary_curated_subcategory"] for row in eligible), 40)


raw rows: 9987
eligible rows: 7572

rejection reasons
 1837  not_in_curated_categories
 1199  directory_like_service_name
  143  missing_address:state_province,zipcode
   61  missing_address:zipcode
   42  missing_core:city
   37  missing_address:address_1
   34  missing_address:address_1,city,state_province,zipcode
   14  missing_contact
    6  missing_address:address_1,state_province,zipcode
    5  missing_address:city
    3  missing_address:city,zipcode
    1  missing_core:site_details
    1  missing_core:site_eligibility

eligible by benchmark category
 1442  Food
 1264  Mental Health & Substance Use
 1166  Health Care
 1056  Benefits & Public Assistance
  808  Family & Community Support
  510  Housing & Shelter
  436  Material Goods
  392  Legal, Employment & Consumer Help
  210  Transportation
  185  Utilities & Financial Assistance
  103  Disaster & Safety Needs

eligible by curated subcategory
 2420  Food
  556  Housing/Shelter
  460  Individual and Family Support Services
  43

## 4. Sampling Strategy

The benchmark should cover many types of needs. A purely random sample will overrepresent common services and large counties. The sampler below is a simple greedy round-robin:

1. Iterate over curated subcategories round by round.
2. Try to take up to `TARGET_PER_SUBCATEGORY` resources per subcategory.
3. Prefer rows from less represented benchmark categories and counties.
4. Cap repeated `service_name` and repeated site to reduce near-duplicates.
5. Cap county count within each benchmark category so Marion/Allen/Lake do not dominate every need type.

This is not mathematically perfect, but it is transparent and easy to tune.


In [5]:
def row_county(row: dict[str, object]) -> str:
    counties = split_multi(str(row.get("counties_served", "")))
    return counties[0] if counties else ""

def row_site_key(row: dict[str, object]) -> tuple[str, str]:
    return (str(row["agency_id"]), str(row["site_id"]))

def candidate_sort_key(
    row: dict[str, object],
    benchmark_counts: Counter,
    county_category_counts: Counter,
    service_counts: Counter,
    site_counts: Counter,
) -> tuple[object, ...]:
    category = str(row["primary_benchmark_category"])
    county = row_county(row)
    service = str(row["service_name"])
    site = row_site_key(row)
    has_fee = 0 if row.get("fee_structure") else 1
    has_docs = 0 if row.get("documents_required") else 1
    return (
        benchmark_counts[category],
        county_category_counts[(category, county)],
        service_counts[service],
        site_counts[site],
        has_docs,
        has_fee,
        str(row["service_name"]),
        str(row["agency_name"]),
        str(row["resource_id"]),
    )

def sample_evenly(rows: list[dict[str, object]]) -> list[dict[str, object]]:
    by_subcategory: dict[str, list[dict[str, object]]] = defaultdict(list)
    for row in rows:
        for subcategory in split_multi(str(row["curated_subcategories"])):
            by_subcategory[subcategory].append(row)

    selected: list[dict[str, object]] = []
    selected_ids: set[str] = set()
    subcategory_counts: Counter = Counter()
    benchmark_counts: Counter = Counter()
    service_counts: Counter = Counter()
    site_counts: Counter = Counter()
    county_category_counts: Counter = Counter()

    subcategories = sorted(by_subcategory, key=lambda sub: (SUBCATEGORY_TO_BENCHMARK_CATEGORY[sub], sub))

    made_progress = True
    while made_progress:
        made_progress = False
        for subcategory in subcategories:
            if subcategory_counts[subcategory] >= TARGET_PER_SUBCATEGORY:
                continue
            candidates = []
            for row in by_subcategory[subcategory]:
                rid = str(row["resource_id"])
                if rid in selected_ids:
                    continue
                service = str(row["service_name"])
                site = row_site_key(row)
                category = SUBCATEGORY_TO_BENCHMARK_CATEGORY[subcategory]
                county = row_county(row)
                if service_counts[service] >= MAX_PER_SERVICE_NAME:
                    continue
                if site_counts[site] >= MAX_PER_SITE:
                    continue
                if county_category_counts[(category, county)] >= MAX_PER_COUNTY_PER_BENCHMARK_CATEGORY:
                    continue
                candidates.append(row)
            if not candidates:
                continue
            picked = min(
                candidates,
                key=lambda row: candidate_sort_key(
                    row, benchmark_counts, county_category_counts, service_counts, site_counts
                ),
            )
            picked = dict(picked)
            picked["sampled_for_subcategory"] = subcategory
            picked["sampled_for_benchmark_category"] = SUBCATEGORY_TO_BENCHMARK_CATEGORY[subcategory]
            selected.append(picked)
            selected_ids.add(str(picked["resource_id"]))
            subcategory_counts[subcategory] += 1
            benchmark_counts[SUBCATEGORY_TO_BENCHMARK_CATEGORY[subcategory]] += 1
            service_counts[str(picked["service_name"])] += 1
            site_counts[row_site_key(picked)] += 1
            county_category_counts[(SUBCATEGORY_TO_BENCHMARK_CATEGORY[subcategory], row_county(picked))] += 1
            made_progress = True
    return selected

sampled = sample_evenly(eligible)
print("sampled rows:", len(sampled))
show_counter("sampled by benchmark category", Counter(row["sampled_for_benchmark_category"] for row in sampled), 20)
show_counter("sampled by source subcategory", Counter(row["sampled_for_subcategory"] for row in sampled), 60)
show_counter("sampled by county", Counter(row_county(row) for row in sampled), 30)
show_counter("sampled top service names", Counter(row["service_name"] for row in sampled), 30)


sampled rows: 636

sampled by benchmark category
  160  Health Care
  108  Family & Community Support
  100  Mental Health & Substance Use
   60  Legal, Employment & Consumer Help
   60  Utilities & Financial Assistance
   38  Disaster & Safety Needs
   30  Benefits & Public Assistance
   20  Food
   20  Housing & Shelter
   20  Material Goods
   20  Transportation

sampled by source subcategory
   20  Public Assistance Programs
   20  Disaster Services
   20  Educational Programs
   20  Educational Support Services
   20  Individual and Family Support Services
   20  Leisure Activities/Recreation
   20  Social Development and Enrichment
   20  Food
   20  Emergency Medical Care
   20  Health Screening/Diagnostic Services
   20  Health Supportive Services
   20  Human Reproduction
   20  Outpatient Health Facilities
   20  Rehabilitation/Habilitation Services
   20  Specialized Treatment and Prevention
   20  Specialty Medicine
   20  Housing/Shelter
   20  Consumer Assistance and Prot

## 5. Write Outputs

- `cleaned_resources_curated.csv`: all rows with pass/fail reasons and curated category labels.
- `benchmark_resource_pool_curated.csv`: sampled benchmark resource pool.
- `resource_index_curated.jsonl`: compact JSONL resource index for retrieval/tool experiments.
- `sampling_report.json`: row counts and distribution summaries.


In [6]:
def to_resource_index_record(row: dict[str, object]) -> dict[str, object]:
    search_fields = [
        "service_name",
        "agency_name",
        "site_name",
        "benchmark_categories",
        "curated_subcategories",
        "taxonomy_categories",
        "subcategories",
        "site_eligibility",
        "agency_desc",
        "site_details",
        "documents_required",
        "fee_structure",
        "city",
        "zipcode",
        "counties_served",
    ]
    return {
        "resource_id": row["resource_id"],
        "agency_id": row["agency_id"],
        "site_id": row["site_id"],
        "service_name": row["service_name"],
        "agency_name": row["agency_name"],
        "site_name": row["site_name"],
        "benchmark_categories": split_multi(str(row["benchmark_categories"])),
        "source_taxonomy_categories": split_multi(str(row["taxonomy_categories"])),
        "source_subcategories": split_multi(str(row["subcategories"])),
        "curated_subcategories": split_multi(str(row["curated_subcategories"])),
        "service_area": split_multi(str(row["counties_served"])),
        "location": {
            "address_1": row["address_1"],
            "address_2": row["address_2"],
            "city": row["city"],
            "state": row["state_province"],
            "zipcode": row["zipcode"],
            "latitude": row["latitude"],
            "longitude": row["longitude"],
        },
        "contact": {
            "phone": row["site_number"],
            "website": row["service_website"],
            "email": row["service_email"],
        },
        "eligibility": row["site_eligibility"],
        "application_process": row["site_details"],
        "fees": row["fee_structure"],
        "documents_required": row["documents_required"],
        "sampled_for_subcategory": row.get("sampled_for_subcategory", ""),
        "sampled_for_benchmark_category": row.get("sampled_for_benchmark_category", ""),
        "search_text": normalize_text(" ".join(str(row.get(field, "")) for field in search_fields)),
    }

write_csv(OUT_DIR / "cleaned_resources_curated.csv", annotated)
write_csv(OUT_DIR / "benchmark_resource_pool_curated.csv", sampled)
write_jsonl(OUT_DIR / "resource_index_curated.jsonl", [to_resource_index_record(row) for row in sampled])

report = {
    "source_csv": str(SOURCE_CSV),
    "raw_rows": len(raw_rows),
    "eligible_rows": len(eligible),
    "sampled_rows": len(sampled),
    "target_per_subcategory": TARGET_PER_SUBCATEGORY,
    "max_per_service_name": MAX_PER_SERVICE_NAME,
    "max_per_site": MAX_PER_SITE,
    "max_per_county_per_benchmark_category": MAX_PER_COUNTY_PER_BENCHMARK_CATEGORY,
    "exclude_directory_like_service_names": EXCLUDE_DIRECTORY_LIKE_SERVICE_NAMES,
    "directory_like_service_names": sorted(DIRECTORY_LIKE_SERVICE_NAMES),
    "curated_category_map": CURATED_CATEGORY_MAP,
    "rejection_reasons": dict(Counter(reason for row in annotated for reason in split_multi(str(row["cleaning_rejection_reasons"])))),
    "eligible_by_benchmark_category": dict(Counter(row["primary_benchmark_category"] for row in eligible)),
    "sampled_by_benchmark_category": dict(Counter(row["sampled_for_benchmark_category"] for row in sampled)),
    "sampled_by_source_subcategory": dict(Counter(row["sampled_for_subcategory"] for row in sampled)),
    "sampled_by_county_top_30": dict(Counter(row_county(row) for row in sampled).most_common(30)),
}
(OUT_DIR / "sampling_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

print("wrote:")
for path in sorted(OUT_DIR.iterdir()):
    print("-", path)


wrote:
- /Users/yiyangli/Projects/211-Agent/data/indiana211/benchmark_curated/benchmark_resource_pool_curated.csv
- /Users/yiyangli/Projects/211-Agent/data/indiana211/benchmark_curated/cleaned_resources_curated.csv
- /Users/yiyangli/Projects/211-Agent/data/indiana211/benchmark_curated/resource_index_curated.jsonl
- /Users/yiyangli/Projects/211-Agent/data/indiana211/benchmark_curated/sampling_report.json


## Next Steps for Query/GT Construction

Use `benchmark_resource_pool_curated.csv` as the seed pool.

Recommended query levels:

1. **Direct need:** county + explicit subcategory, e.g. food pantry, utility help, legal aid.
2. **Constraint need:** add documents, eligibility, fee, age, or application-process hints.
3. **Layperson language:** map phrases like "lights shut off" to Utilities or "nowhere to sleep" to Housing/Shelter.
4. **Near-miss/hard negatives:** same county wrong service, right service wrong county, right category but incompatible eligibility.

GT should usually contain `primary_gt_resource_ids`, `acceptable_gt_resource_ids`, and `hard_negative_resource_ids`, not just one answer.
